# silver_rede_atendimento

Camada **Silver** de rede e atendimento: `ocorrencias_rede`, `atendimentos` e o histórico dos atendimentos.

- **Entrada:** `bronze.tb_ocorrencias_rede`, `bronze.tb_atendimentos` e `silver.tb_unidades_consumidoras`
- **Saída:** `silver.tb_ocorrencias_rede`, `silver.tb_atendimentos`, `silver.tb_atendimentos_historico` e as quarentenas
- **Segue o Padrão de Limpeza:** cada tabela passa pelos 9 passos (ler, estrutura, tipos, domínios, duplicatas, consistência, integridade, checagens, gravar).
- Regras levantadas no notebook `exploracao_rede_atendimento`.

## Funções comuns
Carregando as funções compartilhadas.

In [0]:
%run ./funcoes_comuns

In [0]:
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")
NOTEBOOK = "silver_rede_atendimento"

## Regras de negócio
**O que fazemos:** reunimos os valores aceitos em cada coluna, os limites de duração e a ordem das colunas. Mudar uma regra é mudar só esta célula.

In [0]:
from functools import reduce

# ---------- Ocorrências de rede ----------
TIPOS_OCORRENCIA = ["FALTA DE ENERGIA", "DANO EM EQUIPAMENTO", "OSCILACAO", "SOBRECARGA", "FIO PARTIDO"]
CAUSAS = ["CLIMATICA", "NAO IDENTIFICADA", "SOBRECARGA", "VEGETACAO", "TERCEIROS", "FALHA DE EQUIPAMENTO"]
CAUSA_SEM_VALOR = "NAO INFORMADA"        # campo não preenchido (diferente de NAO IDENTIFICADA)
LIMITE_LONGA_DURACAO_MIN = 1440          # 24h
TETO_DURACAO_MIN = 2880                  # 48h: maior valor da origem, possível teto do sistema

# ---------- Atendimentos ----------
CANAIS = ["TELEFONE", "APP", "WHATSAPP", "SITE", "PRESENCIAL"]
ASSUNTO_SUBASSUNTOS = {
    "DANO ELETRICO":       ["VISTORIA", "RESSARCIMENTO"],
    "FALTA DE ENERGIA":    ["INTERRUPCAO PARCIAL", "INTERRUPCAO TOTAL", "OSCILACAO"],
    "FATURA":              ["CONTESTACAO", "SEGUNDA VIA", "VALOR ALTO", "PARCELAMENTO"],
    "GERACAO DISTRIBUIDA": ["VISTORIA", "HOMOLOGACAO", "CREDITOS"],
    "LIGACAO NOVA":        ["SOLICITACAO", "VISTORIA", "PRAZO"],
    "RELIGACAO":           ["URGENCIA", "APOS PAGAMENTO"],
}
ASSUNTOS = list(ASSUNTO_SUBASSUNTOS)
SUBASSUNTOS = sorted({s for subs in ASSUNTO_SUBASSUNTOS.values() for s in subs})
PARES_ASSUNTO_SUBASSUNTO = [f"{a}|{s}" for a, subs in ASSUNTO_SUBASSUNTOS.items() for s in subs]

# Ordem das colunas (na Bronze o JSON vem em ordem alfabética)
COLUNAS_AT = ["protocolo", "id_uc", "canal", "assunto", "subassunto",
              "data_abertura", "data_fechamento", "nota_satisfacao"]

# ---------- Histórico dos atendimentos ----------
ACOES = ["ABERTURA", "ANALISE", "DESPACHO", "RETORNO AO CLIENTE", "ENCERRAMENTO"]
COLUNAS_HIST = ["protocolo", "sequencia", "data", "acao", "canal"]

## Parte 1: Ocorrências de rede

## Leitura da Bronze
**O que fazemos:** lemos a tabela e guardamos o total de linhas. Esse número é usado no fim para conferir que nenhuma linha sumiu.

In [0]:
oc = ler_bronze(CATALOGO, "ocorrencias_rede")
N_OC_BRONZE = oc.count()
REMOVIDAS_OC = {}
COLUNAS_OC = [c for c in oc.columns if not c.startswith("_")]
print(f"Bronze: {N_OC_BRONZE:,} linhas | colunas: {COLUNAS_OC}")

## Estrutura
**O que fazemos:** removemos cabeçalhos repetidos e tiramos espaços das pontas dos textos. Hoje não há nenhum dos dois casos, mas os passos ficam como proteção.

In [0]:
oc = remover_cabecalho_repetido(oc)                      # proteção: hoje não há cabeçalho repetido
REMOVIDAS_OC["cabecalho"] = N_OC_BRONZE - oc.count()
if REMOVIDAS_OC["cabecalho"]:
    registrar("ocorrencias_rede", "cabeçalho do CSV repetido no meio do arquivo",
              "remover_cabecalho_repetido", "REMOVIDO", REMOVIDAS_OC["cabecalho"])

tem_espaco = reduce(lambda a, b: a | b, [F.col(c).rlike(r"^\s|\s$") for c in COLUNAS_OC])
n_espacos = oc.filter(tem_espaco).count()
oc = aparar_textos(oc)
if n_espacos:
    registrar("ocorrencias_rede", "espaços no início/fim dos textos", "aparar_textos", "CORRIGIDO", n_espacos)
print(f"cabeçalhos removidos: {REMOVIDAS_OC['cabecalho']} | linhas com espaços: {n_espacos}")

## Tipos
**O que fazemos:** convertemos os textos para os tipos certos. Na duração, primeiro removemos o " min" escrito em algumas linhas. O `checar_parse` garante que nenhum valor se perde na conversão.

In [0]:
# Duração com a unidade escrita ("217 min"): remover o texto antes de converter
n_min = oc.filter(F.col("duracao_min").rlike(r"\s*min$")).count()
oc = oc.withColumn("duracao_min", F.regexp_replace("duracao_min", r"\s*min$", ""))

checar_parse(oc, ["duracao_min"], parse_inteiro, "ocorrencias_rede")
checar_parse(oc, ["data_abertura", "data_encerramento"], parse_timestamp, "ocorrencias_rede")

oc = oc.withColumns({
    "duracao_min":       parse_inteiro("duracao_min"),
    "data_abertura":     parse_timestamp("data_abertura"),
    "data_encerramento": parse_timestamp("data_encerramento"),
})
registrar("ocorrencias_rede", "duração com a unidade escrita (ex.: '217 min')",
          "remover ' min' + parse_inteiro", "CORRIGIDO", n_min)

## Domínios
**O que fazemos:** padronizamos as colunas de categoria. A causa sem valor vira NAO INFORMADA, que é diferente de NAO IDENTIFICADA (causa investigada e não encontrada).

In [0]:
# Causa sem valor (nulo, "-", "N/A", "NULL") vira NAO INFORMADA
causa_limpa = padronizar_texto(valores_para_nulo("causa"))
n_causa = oc.filter(causa_limpa.isNull()).count()
oc = oc.withColumn("causa", F.coalesce(causa_limpa, F.lit(CAUSA_SEM_VALOR)))
registrar("ocorrencias_rede", "causa sem valor (nulo, '-', 'N/A', 'NULL')",
          f"valores_para_nulo + {CAUSA_SEM_VALOR}", "CORRIGIDO", n_causa)

# Domínios que já vêm limpos: padronizados mesmo assim, como proteção
oc = oc.withColumns({c: padronizar_texto(c) for c in ["tipo", "alimentador"]})

## Duplicatas
**O que fazemos:** removemos linhas idênticas. Hoje não há nenhuma.

In [0]:
n_antes = oc.count()
oc = deduplicar(oc)
REMOVIDAS_OC["duplicatas"] = n_antes - oc.count()
if REMOVIDAS_OC["duplicatas"]:
    registrar("ocorrencias_rede", "linhas duplicadas idênticas", "deduplicar(df)", "REMOVIDO",
              REMOVIDAS_OC["duplicatas"])
print(f"Antes: {n_antes:,} | duplicatas removidas: {REMOVIDAS_OC['duplicatas']:,}")

## Consistência
**O que fazemos:** marcamos com flags os casos suspeitos, sem apagar nenhuma linha. São as interrupções acima de 24h, no teto de 48h e sobrepostas na mesma UC. Também marcamos incoerências de tempo, que hoje não existem. Cada flag é registrada no log.

In [0]:
ab, enc, dur = F.col("data_abertura"), F.col("data_encerramento"), F.col("duracao_min")
dur_calculada = (F.unix_timestamp(enc) - F.unix_timestamp(ab)) / 60

# Sobreposição: começa antes do fim mais tardio de todas as interrupções anteriores da mesma UC
w_anteriores = (Window.partitionBy("id_uc").orderBy("data_abertura", "id_ocorrencia")
                      .rowsBetween(Window.unboundedPreceding, -1))

oc = criar_flag(oc, "flag_longa_duracao", dur > LIMITE_LONGA_DURACAO_MIN)
oc = criar_flag(oc, "flag_duracao_teto", dur >= TETO_DURACAO_MIN)
oc = criar_flag(oc, "flag_sobreposicao", ab < F.max("data_encerramento").over(w_anteriores))
oc = criar_flag(oc, "flag_duracao_inconsistente", dur != dur_calculada)
oc = criar_flag(oc, "flag_encerramento_antes_abertura", enc <= ab)

FLAGS_OC = {
    "flag_longa_duracao": ("interrupção acima de 24h", f"duracao_min > {LIMITE_LONGA_DURACAO_MIN}"),
    "flag_duracao_teto": ("interrupção no teto de 48h da origem (duração real pode ter sido maior)",
                          f"duracao_min >= {TETO_DURACAO_MIN}"),
    "flag_sobreposicao": ("interrupção sobreposta a outra da mesma UC (tempo contado em dobro no DEC)",
                          "abertura < maior encerramento anterior da UC"),
    "flag_duracao_inconsistente": ("duração diferente do tempo entre abertura e encerramento",
                                   "duracao_min != encerramento - abertura"),
    "flag_encerramento_antes_abertura": ("encerramento antes ou igual à abertura",
                                         "data_encerramento <= data_abertura"),
}
contagem = oc.agg(*[F.sum(F.col(f).cast("int")).alias(f) for f in FLAGS_OC]).first().asDict()
for flag, (problema, regra) in FLAGS_OC.items():
    registrar("ocorrencias_rede", problema, regra, "FLAG", contagem[flag])

## Integridade com o cadastro
**O que fazemos:** marcamos as ocorrências com alimentador diferente do cadastro e mandamos para a quarentena as de UCs que não existem. Hoje são zero nos dois casos.

In [0]:
uc_silver = spark.table(f"{CATALOGO}.silver.tb_unidades_consumidoras")

# Alimentador da ocorrência diferente do alimentador da UC no cadastro
alim_cad = uc_silver.select("id_uc", F.col("alimentador").alias("__alimentador_cad"))
oc = oc.join(alim_cad, "id_uc", "left")
oc = criar_flag(oc, "flag_alimentador_inconsistente",
                F.col("alimentador") != F.col("__alimentador_cad")).drop("__alimentador_cad")
FLAGS_OC["flag_alimentador_inconsistente"] = ("alimentador diferente do alimentador da UC no cadastro",
                                              "alimentador != silver.tb_unidades_consumidoras.alimentador")
registrar("ocorrencias_rede", *FLAGS_OC["flag_alimentador_inconsistente"], "FLAG",
          oc.filter("flag_alimentador_inconsistente").count())

# Órfãos (último passo de transformação)
oc, oc_quarentena = separar_orfaos(oc, uc_silver, "id_uc",
                                   motivo="id_uc inexistente em silver.tb_unidades_consumidoras")
registrar("ocorrencias_rede", "ocorrência de UC inexistente no cadastro",
          "separar_orfaos vs silver.tb_unidades_consumidoras", "QUARENTENA", oc_quarentena.count())

## Checagens
**O que fazemos:** conferimos chave única, colunas sem nulos e valores dentro do esperado. A reconciliação confere que Bronze é igual a Silver mais quarentena mais removidas. Se algo falhar, o notebook para aqui e a Silver antiga fica intacta.

In [0]:
checar_unicidade(oc, ["id_ocorrencia"], "ocorrencias_rede")
checar_sem_nulos(oc, COLUNAS_OC, "ocorrencias_rede")
checar_dominio(oc, "tipo", TIPOS_OCORRENCIA, "ocorrencias_rede")
checar_dominio(oc, "causa", CAUSAS + [CAUSA_SEM_VALOR], "ocorrencias_rede")

reconciliar("ocorrencias_rede", N_OC_BRONZE, oc.count(), oc_quarentena.count(), REMOVIDAS_OC)

## Gravação
**O que fazemos:** organizamos as colunas na ordem padrão (negócio, flags, rastreio) e gravamos a Silver e a quarentena.

In [0]:
oc = oc.select(*COLUNAS_OC, *FLAGS_OC, "_source_file", "_ingestion_timestamp")
n_oc = gravar_silver(oc, CATALOGO, "ocorrencias_rede", quarentena=oc_quarentena)
spark.table(f"{CATALOGO}.silver.tb_ocorrencias_rede").printSchema()

## Parte 2: Atendimentos

## Leitura da Bronze
**O que fazemos:** lemos a tabela e guardamos o total de linhas. O histórico é separado aqui e vira uma tabela própria na Parte C.

In [0]:
at_bronze = ler_bronze(CATALOGO, "atendimentos")
N_AT_BRONZE = at_bronze.count()
REMOVIDAS_AT = {}
at = at_bronze.drop("historico")          # o histórico vira a tabela filha (Parte C)
print(f"Bronze: {N_AT_BRONZE:,} linhas")

## Estrutura e tipos
**O que fazemos:** tiramos espaços das pontas e convertemos os tipos. As datas viram data e hora, e a nota vira decimal com uma casa. A nota vazia continua nula, nunca zero.

In [0]:
# Estrutura
at = remover_cabecalho_repetido(at)                      # proteção
REMOVIDAS_AT["cabecalho"] = N_AT_BRONZE - at.count()
tem_espaco_at = reduce(lambda a, b: a | b, [F.col(c).rlike(r"^\s|\s$") for c in COLUNAS_AT])
n_espacos_at = at.filter(tem_espaco_at).count()
at = aparar_textos(at)
if n_espacos_at:
    registrar("atendimentos", "espaços no início/fim dos textos", "aparar_textos", "CORRIGIDO", n_espacos_at)

# Tipos
checar_parse(at, ["data_abertura", "data_fechamento"], parse_timestamp, "atendimentos")
checar_parse(at, ["nota_satisfacao"], parse_decimal, "atendimentos", precisao=3, escala=1)
at = at.withColumns({
    "data_abertura":   parse_timestamp("data_abertura"),
    "data_fechamento": parse_timestamp("data_fechamento"),
    "nota_satisfacao": parse_decimal("nota_satisfacao", 3, 1),
})
registrar("atendimentos", "nota de satisfação não preenchida", "mantida nula (nunca zero)", "MANTIDO",
          at.filter(F.col("nota_satisfacao").isNull()).count())
print(f"cabeçalhos: {REMOVIDAS_AT['cabecalho']} | linhas com espaços: {n_espacos_at}")

## Domínios e duplicatas
**O que fazemos:** padronizamos canal, assunto e subassunto como proteção e removemos linhas idênticas. Hoje não há nenhuma.

In [0]:
at = at.withColumns({c: padronizar_texto(c) for c in ["canal", "assunto", "subassunto"]})   # proteção

n_antes = at.count()
at = deduplicar(at)
REMOVIDAS_AT["duplicatas"] = n_antes - at.count()
if REMOVIDAS_AT["duplicatas"]:
    registrar("atendimentos", "linhas duplicadas idênticas", "deduplicar(df)", "REMOVIDO",
              REMOVIDAS_AT["duplicatas"])
print(f"Antes: {n_antes:,} | duplicatas removidas: {REMOVIDAS_AT['duplicatas']:,}")

## Consistência e integridade
**O que fazemos:** marcamos com flags o fechamento antes da abertura e o subassunto que não pertence ao assunto. Depois mandamos para a quarentena os atendimentos de UCs que não existem no cadastro. Hoje são zero em todos os casos.

In [0]:
at = criar_flag(at, "flag_fechamento_antes_abertura", F.col("data_fechamento") < F.col("data_abertura"))
at = criar_flag(at, "flag_assunto_subassunto_inconsistente",
                ~F.concat_ws("|", "assunto", "subassunto").isin(PARES_ASSUNTO_SUBASSUNTO))

FLAGS_AT = {
    "flag_fechamento_antes_abertura": ("fechamento antes da abertura", "data_fechamento < data_abertura"),
    "flag_assunto_subassunto_inconsistente": ("subassunto que não pertence ao assunto", "ASSUNTO_SUBASSUNTOS"),
}
contagem = at.agg(*[F.sum(F.col(f).cast("int")).alias(f) for f in FLAGS_AT]).first().asDict()
for flag, (problema, regra) in FLAGS_AT.items():
    registrar("atendimentos", problema, regra, "FLAG", contagem[flag])

at, at_quarentena = separar_orfaos(at, uc_silver, "id_uc",
                                   motivo="id_uc inexistente em silver.tb_unidades_consumidoras")
registrar("atendimentos", "atendimento de UC inexistente no cadastro",
          "separar_orfaos vs silver.tb_unidades_consumidoras", "QUARENTENA", at_quarentena.count())

## Checagens e gravação
**O que fazemos:** rodamos as checagens finais e a reconciliação. Se tudo passar, organizamos as colunas e gravamos a Silver e a quarentena.

In [0]:
checar_unicidade(at, ["protocolo"], "atendimentos")
checar_sem_nulos(at, [c for c in COLUNAS_AT if c != "nota_satisfacao"], "atendimentos")
checar_dominio(at, "canal", CANAIS, "atendimentos")
checar_dominio(at, "assunto", ASSUNTOS, "atendimentos")
checar_dominio(at, "subassunto", SUBASSUNTOS, "atendimentos")

reconciliar("atendimentos", N_AT_BRONZE, at.count(), at_quarentena.count(), REMOVIDAS_AT)

at = at.select(*COLUNAS_AT, *FLAGS_AT, "_source_file", "_ingestion_timestamp")
n_at = gravar_silver(at, CATALOGO, "atendimentos", quarentena=at_quarentena)
spark.table(f"{CATALOGO}.silver.tb_atendimentos").printSchema()

## Parte 3: Histórico dos atendimentos
Tabela filha com uma linha por evento do histórico. A chave é protocolo mais sequência.

## Abertura da lista, estrutura e tipos
**O que fazemos:** abrimos a lista de eventos de cada atendimento, uma linha por evento. Depois tiramos espaços, convertemos a sequência para número e a data para data e hora.

In [0]:
# Uma linha por evento, a partir da Bronze (todos os protocolos; os órfãos saem no fim)
hist = (at_bronze
        .select("protocolo", F.explode("historico").alias("e"), "_source_file", "_ingestion_timestamp")
        .select("protocolo",
                F.col("e.sequencia").alias("sequencia"), F.col("e.data").alias("data"),
                F.col("e.acao").alias("acao"), F.col("e.canal").alias("canal"),
                "_source_file", "_ingestion_timestamp"))
N_HIST_BRONZE = hist.count()
REMOVIDAS_HIST = {}
registrar("atendimentos_historico", "histórico aninhado em lista dentro do atendimento",
          "explode em tabela filha (uma linha por evento)", "CORRIGIDO", N_HIST_BRONZE)
print(f"Eventos: {N_HIST_BRONZE:,}")

hist = aparar_textos(hist)
checar_parse(hist, ["sequencia"], parse_inteiro, "atendimentos_historico")
checar_parse(hist, ["data"], parse_timestamp, "atendimentos_historico")
hist = hist.withColumns({"sequencia": parse_inteiro("sequencia"), "data": parse_timestamp("data")})
hist = hist.withColumns({c: padronizar_texto(c) for c in ["acao", "canal"]})   # proteção

## Duplicatas e consistência
**O que fazemos:** removemos eventos idênticos e conferimos cada evento contra o seu atendimento, já gravado na Silver. Verificamos se a sequência não tem buracos, se a data está dentro do período do atendimento e se o canal é o mesmo. A ordem das ações não faz sentido de negócio, mas fica como está e é registrada no log.

In [0]:
n_antes = hist.count()
hist = deduplicar(hist)
REMOVIDAS_HIST["duplicatas"] = n_antes - hist.count()
if REMOVIDAS_HIST["duplicatas"]:
    registrar("atendimentos_historico", "eventos duplicados idênticos", "deduplicar(df)", "REMOVIDO",
              REMOVIDAS_HIST["duplicatas"])

# Coerência com o atendimento (lido da Silver já gravada)
at_silver = spark.table(f"{CATALOGO}.silver.tb_atendimentos")
ref_at = at_silver.select("protocolo", F.col("data_abertura").alias("__ab"),
                          F.col("data_fechamento").alias("__fe"), F.col("canal").alias("__canal"))
hist = hist.join(ref_at, "protocolo", "left")
w_prot = Window.partitionBy("protocolo").orderBy("sequencia")

hist = criar_flag(hist, "flag_sequencia_inconsistente", F.col("sequencia") != F.row_number().over(w_prot))
hist = criar_flag(hist, "flag_evento_fora_atendimento",
                  (F.col("data") < F.col("__ab")) | (F.col("data") > F.col("__fe")))
hist = criar_flag(hist, "flag_canal_diferente_atendimento", F.col("canal") != F.col("__canal"))
hist = hist.drop("__ab", "__fe", "__canal")

FLAGS_HIST = {
    "flag_sequencia_inconsistente": ("sequência com buraco ou sem começar em 1", "sequencia != posição 1..n"),
    "flag_evento_fora_atendimento": ("evento fora do período do atendimento", "data fora de [abertura, fechamento]"),
    "flag_canal_diferente_atendimento": ("canal do evento diferente do canal do atendimento", "canal != atendimento.canal"),
}
contagem = hist.agg(*[F.sum(F.col(f).cast("int")).alias(f) for f in FLAGS_HIST]).first().asDict()
for flag, (problema, regra) in FLAGS_HIST.items():
    registrar("atendimentos_historico", problema, regra, "FLAG", contagem[flag])

# Ordem das ações: problema conhecido, mantido como está
n_ordem = hist.filter((F.col("sequencia") == 1) & (F.col("acao") != "ABERTURA")).count()
registrar("atendimentos_historico", "ordem das ações sem sentido de negócio (1º evento nem sempre é ABERTURA)",
          "mantido; não usar acao para métricas", "MANTIDO", n_ordem)

## Integridade, checagens e gravação
**O que fazemos:** mandamos para a quarentena os eventos de atendimentos que não estão na Silver. Depois rodamos as checagens finais, a reconciliação e gravamos.

In [0]:
hist, hist_quarentena = separar_orfaos(hist, at_silver, "protocolo",
                                       motivo="protocolo inexistente em silver.tb_atendimentos")
registrar("atendimentos_historico", "evento de atendimento inexistente na Silver",
          "separar_orfaos vs silver.tb_atendimentos", "QUARENTENA", hist_quarentena.count())

checar_unicidade(hist, ["protocolo", "sequencia"], "atendimentos_historico")
checar_sem_nulos(hist, COLUNAS_HIST, "atendimentos_historico")
checar_dominio(hist, "acao", ACOES, "atendimentos_historico")
checar_dominio(hist, "canal", CANAIS, "atendimentos_historico")

reconciliar("atendimentos_historico", N_HIST_BRONZE, hist.count(), hist_quarentena.count(), REMOVIDAS_HIST)

hist = hist.select(*COLUNAS_HIST, *FLAGS_HIST, "_source_file", "_ingestion_timestamp")
n_hist = gravar_silver(hist, CATALOGO, "atendimentos_historico", quarentena=hist_quarentena)
spark.table(f"{CATALOGO}.silver.tb_atendimentos_historico").printSchema()

## Fechamento: log de qualidade
**O que fazemos:** gravamos no log todos os problemas tratados neste notebook e mostramos o resultado.

In [0]:
gravar_log(CATALOGO, NOTEBOOK)

(spark.table(f"{CATALOGO}.silver.tb_log_qualidade")
      .filter(F.col("notebook") == NOTEBOOK)
      .select("tabela", "problema", "destino", "linhas")
      .orderBy("tabela", "destino")
      .show(100, truncate=False))